# Member 2 — Heuristic CVRP Methods | Version 1: Clarke–Wright Savings
**Git commit:** `feat(heuristic): implement Clarke-Wright savings construction`

This version builds a feasible route plan quickly. It uses exactly the same deterministic instance and distance convention as Member 1. Validate every returned solution before reporting its objective.

## 1. Dependencies and shared instance

In [ ]:
!pip -q install numpy pandas matplotlib

In [ ]:
import math, random, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
N_CUSTOMERS = 20   # Increase to 31 to match the scale of the sample report
K = 5
CAPACITY = 100
rng = np.random.default_rng(SEED)

# Shared deterministic synthetic instance. Both members must use these same parameters.
coords = [(50.0, 50.0)] + [(float(x), float(y)) for x,y in rng.uniform(0,100,size=(N_CUSTOMERS,2))]
demands = [0] + [int(x) for x in rng.integers(4, 11, size=N_CUSTOMERS)]
n = len(coords)
dist = np.zeros((n,n), dtype=int)
for i in range(n):
    for j in range(n):
        if i != j:
            dist[i,j] = int(math.floor(math.dist(coords[i], coords[j]) + 0.5))

print(f"Nodes: {n} (including depot 0); customers: {N_CUSTOMERS}; vehicles: {K}; capacity: {CAPACITY}")
print(f"Total demand: {sum(demands)}; total fleet capacity: {K*CAPACITY}")
assert sum(demands) <= K*CAPACITY, "Infeasible generated instance; adjust seed/demands."

In [ ]:
def route_distance(route):
    """Distance for a customer list; depot 0 is added at both ends."""
    seq = [0] + list(route) + [0]
    return int(sum(dist[seq[i], seq[i+1]] for i in range(len(seq)-1)))

def solution_cost(routes):
    return sum(route_distance(r) for r in routes)

def validate_routes(routes, verbose=True):
    errors = []
    seen = [c for route in routes for c in route]
    if sorted(seen) != list(range(1,n)):
        errors.append("Every customer must appear exactly once.")
    if len(routes) > K:
        errors.append(f"Used {len(routes)} routes, exceeding fleet limit {K}.")
    for ridx, route in enumerate(routes, start=1):
        load = sum(demands[c] for c in route)
        if load > CAPACITY:
            errors.append(f"Route {ridx} load {load} exceeds capacity {CAPACITY}.")
        if not route:
            errors.append(f"Route {ridx} is empty.")
    cost = solution_cost(routes)
    if verbose:
        print("Feasible:", not errors, "| Cost:", cost, "| Routes:", len(routes))
        for e in errors: print(" -", e)
    return {"feasible": not errors, "cost": cost, "routes_used": len(routes), "errors": errors}

## 2. Clarke–Wright Savings algorithm

In [ ]:
def clarke_wright_savings():
    # Initially, each customer is served by its own depot-customer-depot route.
    routes = {i:[i] for i in range(1,n)}
    route_of = {i:i for i in range(1,n)}
    savings=[]
    for i in range(1,n):
        for j in range(i+1,n):
            savings.append((int(dist[0,i]+dist[0,j]-dist[i,j]),i,j))
    savings.sort(reverse=True)

    for saving,i,j in savings:
        ri, rj = route_of.get(i), route_of.get(j)
        if ri is None or rj is None or ri == rj:
            continue
        a,b=routes[ri],routes[rj]
        # Merge only if i and j are route endpoints; orient so i is at end and j at start.
        if a[-1] != i:
            if a[0] == i: a=list(reversed(a))
            else: continue
        if b[0] != j:
            if b[-1] == j: b=list(reversed(b))
            else: continue
        if sum(demands[c] for c in a+b) > CAPACITY:
            continue
        routes[ri]=a+b
        del routes[rj]
        for c in routes[ri]: route_of[c]=ri

    result=[r for r in routes.values() if r]
    # If this merge order leaves more than K routes, try further savings merges already
    # exhausted; flag infeasibility rather than silently claiming success.
    validation=validate_routes(result)
    return result

cw_routes=clarke_wright_savings()
cw_validation=validate_routes(cw_routes)
print("Route loads:", [sum(demands[c] for c in r) for r in cw_routes])

## 3. Plot the routes

In [ ]:
def plot_routes(routes, title="CVRP routes"):
    plt.figure(figsize=(8,6))
    plt.scatter([coords[i][0] for i in range(1,n)],[coords[i][1] for i in range(1,n)],label="Customers")
    plt.scatter([coords[0][0]],[coords[0][1]],marker="s",s=120,label="Depot")
    for i in range(1,n):
        plt.annotate(str(i),(coords[i][0],coords[i][1]),fontsize=8)
    for route in routes:
        seq=[0]+route+[0]
        plt.plot([coords[i][0] for i in seq],[coords[i][1] for i in seq],marker="o",alpha=.8)
    plt.title(f"{title} | distance={solution_cost(routes)}")
    plt.xlabel("X"); plt.ylabel("Y"); plt.legend(); plt.grid(True); plt.show()

plot_routes(cw_routes,"Clarke–Wright Savings")

## Version 1 checklist
- [ ] Confirm all customers are served exactly once.
- [ ] Confirm no route exceeds capacity and route count is at most K.
- [ ] Record route loads, total distance, and runtime.
- [ ] Commit the working construction before adding Simulated Annealing.